In [2]:
%pip install mlflow dvc  


  Using cached mlflow-3.17.0-py3-none-any.whl.metadata (50 kB)
  Using cached dvc-3.67.1-py3-none-any.whl.metadata (17 kB)
  Using cached mlflow_skinny-3.17.0-py3-none-any.whl.metadata (51 kB)
  Using cached mlflow_tracing-3.17.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.5-py3-none-any.whl.metadata (5.4 kB)
  Using cached aiohttp-3.14.5-cp313-cp313-win_amd64.whl.metadata (8.5 kB)
  Using cached alembic-1.20.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached cryptography-50.0.2-cp311-abi3-win_amd64.whl.metadata (4.4 kB)
  Using cached docker-7.2.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached skops-0.16.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached cachetools-7.2.1-py3-none-any.whl.metadata (5.5 kB)
  Using cached databricks_sdk-0.151.0-py3-none-any.whl.metadata (44 kB)
  Using cached fastapi-0.143.0-py3-none-any.whl.metadata (26 kB)
  Using cached gitpython-3.2.0-py3-none-any.whl.metad

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Users\\madhu\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\pyasn1_modules\\rfc7508.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install dvc

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import mlflow
import dvc
import sklearn
import pandas
import joblib
import matplotlib

print("MLflow version:", mlflow.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("All libraries imported successfully!")

MLflow version: 3.17.0
Scikit-learn version: 1.9.0
All libraries imported successfully!


In [4]:
%pip show dvc

Name: dvc
Version: 3.67.1
Summary: Git for data scientists - manage your code and data together
Home-page: 
Author: 
Author-email: Dmitry Petrov <dmitry@dvc.org>
License-Expression: Apache-2.0
Location: c:\Users\madhu\AppData\Local\Programs\Python\Python313\Lib\site-packages
Requires: attrs, celery, colorama, configobj, distro, dpath, dulwich, dvc-data, dvc-http, dvc-objects, dvc-render, dvc-studio-client, dvc-task, flatten-dict, flufl.lock, fsspec, funcy, grandalf, gto, hydra-core, iterative-telemetry, kombu, networkx, omegaconf, packaging, pathspec, platformdirs, psutil, pydot, pygtrie, pyparsing, requests, rich, ruamel.yaml, scmrepo, shortuuid, shtab, tabulate, tomlkit, tqdm, voluptuous, zc.lockfile
Required-by: 
Note: you may need to restart the kernel to use updated packages.


# dataset generate

In [6]:
import os
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Create project folders
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("screenshots", exist_ok=True)

# Load dataset
dataset = load_breast_cancer(as_frame=True)

# Combine features and target
df = dataset.frame

# Save dataset
df.to_csv("data/dataset.csv", index=False)

print("Dataset created successfully!")
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Dataset created successfully!
Dataset shape: (569, 31)

First 5 rows:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


# : Train 3 models using MLflow

In [8]:
import os
import mlflow
import mlflow.sklearn
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load dataset
df = pd.read_csv("data/dataset.csv")

X = df.drop(columns=["target"])
y = df["target"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create models folder
os.makedirs("models", exist_ok=True)

# Configure MLflow experiment
mlflow.set_experiment("MLTrackPro_Classification")

experiments = [
    {
        "name": "Logistic_Regression",
        "model": make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, C=1.0)
        ),
        "params": {"model": "LogisticRegression", "C": 1.0}
    },
    {
        "name": "Random_Forest_50",
        "model": RandomForestClassifier(
            n_estimators=50, max_depth=10, random_state=42
        ),
        "params": {
            "model": "RandomForest",
            "n_estimators": 50,
            "max_depth": 10
        }
    },
    {
        "name": "SVM_RBF",
        "model": make_pipeline(
            StandardScaler(),
            SVC(C=1.0, kernel="rbf")
        ),
        "params": {
            "model": "SVC",
            "C": 1.0,
            "kernel": "rbf"
        }
    }
]

results = []

for exp in experiments:
    with mlflow.start_run(run_name=exp["name"]):

        model = exp["model"]
        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        accuracy = accuracy_score(y_test, predictions)
        precision = precision_score(y_test, predictions, zero_division=0)
        recall = recall_score(y_test, predictions, zero_division=0)
        f1 = f1_score(y_test, predictions, zero_division=0)

        # Log parameters
        mlflow.log_params(exp["params"])

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        # Save model artifact in MLflow
        mlflow.sklearn.log_model(
            sk_model=model,
            name="model",
            serialization_format="pickle"
       )

        # Save model file for DVC versioning
        model_path = f"models/{exp['name']}.joblib"
        joblib.dump(model, model_path)

        results.append({
            "Experiment": exp["name"],
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1
        })

# Display experiment comparison
results_df = pd.DataFrame(results)
display(results_df.round(4))

print("All 3 experiments logged successfully!")
print("Trained model files saved in models/")

2026/10/12 20:40:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/12 20:40:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/12 20:40:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

,Experiment,Accuracy,Precision,Recall,F1 Score
0,Logistic_Regression,0.9825,0.9861,0.9861,0.9861
1,Random_Forest_50,0.9561,0.9589,0.9722,0.9655
2,SVM_RBF,0.9825,0.9861,0.9861,0.9861


All 3 experiments logged successfully!
Trained model files saved in models/


# 3 MLflow experiments successfully run 

| Model               | Accuracy | F1 Score |
| ------------------- | -------: | -------: |
| Logistic Regression |   98.25% |   98.61% |
| Random Forest       |   95.61% |   96.55% |
| SVM (RBF)           |   98.25% |   98.61% |


# MLflow UI

In [9]:
import mlflow

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name("MLTrackPro_Classification").name)

Tracking URI: sqlite:///C:/Users/madhu/OneDrive/Desktop/6%20month%20internship%20program/MLTrackPro/mlflow.db
Experiment: MLTrackPro_Classification


# DVC setup

In [10]:
import subprocess

result = subprocess.run(
    ["dvc", "init"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


ERROR: failed to initiate DVC - c:\Users\madhu\OneDrive\Desktop\6 month internship program\MLTrackPro is not tracked by any supported SCM tool (e.g. Git). Use `--no-scm` if you don't want to use any SCM or `--subdir` if initializing inside a subdirectory of a parent SCM repository.



# Git initialize

In [11]:
import subprocess

result = subprocess.run(
    ["git", "init"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

Initialized empty Git repository in C:/Users/madhu/OneDrive/Desktop/6 month internship program/MLTrackPro/.git/




# DVC initialize

In [12]:
import subprocess

result = subprocess.run(
    ["dvc", "init"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>




# Track the Dataset with DVC

In [13]:
import subprocess

result = subprocess.run(
    ["dvc", "add", "data/dataset.csv"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


To track the changes with git, run:

	git add 'data\dataset.csv.dvc' 'data\.gitignore'

To enable auto staging, run:

	dvc config core.autostage true

â ‹ Checking graph




# Track the Trained Models with DVC

In [14]:
import subprocess

result = subprocess.run(
    ["dvc", "add", "models/"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


To track the changes with git, run:

	git add .gitignore models.dvc

To enable auto staging, run:

	dvc config core.autostage true

â ‹ Checking graph




# Configure a Local DVC Remote

In [15]:
import os
import subprocess

os.makedirs("../MLTrackPro_DVC_Remote", exist_ok=True)

result = subprocess.run(
    ["dvc", "remote", "add", "-d", "localremote", "../MLTrackPro_DVC_Remote"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

Setting 'localremote' as a default remote.




# Push the Dataset and Models to DVC Remote

In [16]:
import subprocess

result = subprocess.run(
    ["dvc", "push"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

5 files pushed




# Check DVC Status

In [17]:
import subprocess

result = subprocess.run(
    ["dvc", "status"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

Data and pipelines are up to date.




# Stage and Commit Files to Git

In [18]:
import subprocess

commands = [
    ["git", "add", "."],
    ["git", "status", "--short"]
]

for command in commands:
    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )
    print("$", " ".join(command))
    print(result.stdout)
    print(result.stderr)

$ git add .


$ git status --short
A  .dvc/.gitignore
A  .dvc/config
A  .dvcignore
A  .gitignore
A  data/.gitignore
A  data/dataset.csv.dvc
A  experiment_tracking.ipynb
A  mlflow.db
A  mlruns/1/models/m-170215c59be04b4caaa4e0de49fa692f/artifacts/MLmodel
A  mlruns/1/models/m-170215c59be04b4caaa4e0de49fa692f/artifacts/conda.yaml
A  mlruns/1/models/m-170215c59be04b4caaa4e0de49fa692f/artifacts/model.skops
A  mlruns/1/models/m-170215c59be04b4caaa4e0de49fa692f/artifacts/python_env.yaml
A  mlruns/1/models/m-170215c59be04b4caaa4e0de49fa692f/artifacts/requirements.txt
A  mlruns/1/models/m-61f0b218e3194614a9d89368602da47e/artifacts/MLmodel
A  mlruns/1/models/m-61f0b218e3194614a9d89368602da47e/artifacts/conda.yaml
A  mlruns/1/models/m-61f0b218e3194614a9d89368602da47e/artifacts/model.pkl
A  mlruns/1/models/m-61f0b218e3194614a9d89368602da47e/artifacts/python_env.yaml
A  mlruns/1/models/m-61f0b218e3194614a9d89368602da47e/artifacts/requirements.txt
A  mlruns/1/models/m-f2fad98b5c994600ae24d696e210fe1

| Task                                | Status     |
| ----------------------------------- | ---------- |
| Created 3 MLflow experiments        | ✅ Complete |
| Logged model parameters and metrics | ✅ Complete |
| Opened the MLflow UI                | ✅ Complete |
| Initialized Git and DVC             | ✅ Complete |
| Tracked the dataset with DVC        | ✅ Complete |
| Tracked trained models with DVC     | ✅ Complete |
| Configured a local DVC remote       | ✅ Complete |
| Pushed 5 files to the DVC remote    | ✅ Complete |
| Committed project files to Git      | ✅ Complete |
| Pushed the project to GitHub        | ✅ Complete |


In [19]:
import mlflow
import pandas as pd

# Retrieve the previous MLflow run
run_id = "0ef65e0a6a504df69a16800c866baee3"
client = mlflow.tracking.MlflowClient()
run = client.get_run(run_id)

# Display the logged parameters and metrics
print("Previous Experiment Parameters:")
print(run.data.params)

print("\nPrevious Experiment Metrics:")
print(run.data.metrics)

# Load the DVC-tracked dataset
df = pd.read_csv("data/dataset.csv")

print("\nDataset loaded successfully!")
print("Dataset shape:", df.shape)

Previous Experiment Parameters:
{'model': 'LogisticRegression', 'C': '1.0'}

Previous Experiment Metrics:
{'accuracy': 0.9824561403508771, 'precision': 0.9861111111111112, 'recall': 0.9861111111111112, 'f1_score': 0.9861111111111112}

Dataset loaded successfully!
Dataset shape: (569, 31)


# Reproduce Logistic Regression

In [20]:
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Retrieve parameters from the original MLflow run
run_id = "0ef65e0a6a504df69a16800c866baee3"
client = mlflow.tracking.MlflowClient()
original_run = client.get_run(run_id)

C_value = float(original_run.data.params["C"])

# Load the DVC-tracked dataset
df = pd.read_csv("data/dataset.csv")
X = df.drop(columns=["target"])
y = df["target"]

# Recreate the original train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train using the logged parameter
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, C=C_value)
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

# Calculate evaluation metrics
metrics = {
    "accuracy": accuracy_score(y_test, predictions),
    "precision": precision_score(y_test, predictions, zero_division=0),
    "recall": recall_score(y_test, predictions, zero_division=0),
    "f1_score": f1_score(y_test, predictions, zero_division=0)
}

# Log the reproduced experiment as a new MLflow run
with mlflow.start_run(run_name="Logistic_Regression_Reproduced"):
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("C", C_value)
    mlflow.log_param("reproduced_from_run", run_id)
    mlflow.log_metrics(metrics)

    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        serialization_format="pickle"
    )

print("Original run ID:", run_id)
print("Reproduced parameters:", {"C": C_value})
print("\nReproduced metrics:")

for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

print("\nExperiment reproduction completed successfully!")

2026/10/12 22:16:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Original run ID: 0ef65e0a6a504df69a16800c866baee3
Reproduced parameters: {'C': 1.0}

Reproduced metrics:
accuracy: 0.9825
precision: 0.9861
recall: 0.9861
f1_score: 0.9861

Experiment reproduction completed successfully!


In [ ]:
import mlflow
import pandas as pd

# Retrieve the previous MLflow run
run_id = "0ef65e0a6a504df69a16800c866baee3"
client = mlflow.tracking.MlflowClient()
run = client.get_run(run_id)

# Display the logged parameters and metrics
print("Previous Experiment Parameters:")
print(run.data.params)

print("\nPrevious Experiment Metrics:")
print(run.data.metrics)

# Load the DVC-tracked dataset
df = pd.read_csv("data/dataset.csv")

print("\nDataset loaded successfully!")
print("Dataset shape:", df.shape)

Previous Experiment Parameters:
{'model': 'LogisticRegression', 'C': '1.0'}

Previous Experiment Metrics:
{'accuracy': 0.9824561403508771, 'precision': 0.9861111111111112, 'recall': 0.9861111111111112, 'f1_score': 0.9861111111111112}

Dataset loaded successfully!
Dataset shape: (569, 31)
